### Contexto del problema a resolver:

La Clínica San Gabriel llevaba el control de sus citas en cuadernos, lo que producía tres problemas: se programaban **dos pacientes con el mismo médico a la misma hora**, el **historial clínico se perdía** y **no existían reportes**.

El sistema resuelve los tres modelando cada elemento del proceso como un objeto: `Paciente`, `Medico`, `Cita`, `Historial` y `Clinica`, todas construidas sobre la clase base `Persona`.


---

# Parte 1 — Anaya Yarleque, Nahomi Alessandra

Las clases base del sistema. Aquí se definen los datos que comparten todas las personas y la estructura que guarda los antecedentes médicos.

## 1.1 Clase `Historial`

Guarda las atenciones de un paciente. Cada atención es un **diccionario**, para poder convertirlas después en una tabla de pandas sin trabajo adicional.

Se define primero porque `Paciente` la necesita: al crear un paciente se le crea también su historial.

In [ ]:
# ------------------------------------------------------------
#  Clase Historial
#  Integrante: Anaya Yarleque, Nahomi Alessandra
# ------------------------------------------------------------
# Guarda las atenciones de UN paciente. Cada atencion se guarda
# como un diccionario, para poder convertirlas despues en una
# tabla de pandas sin trabajo adicional.


class Historial:
    # Atributo de CLASE: el mismo para todos los historiales
    centro = "Clinica San Gabriel"

    def __init__(self, codigo_paciente):
        # Atributos de INSTANCIA: distintos en cada objeto
        self.codigo_paciente = codigo_paciente
        self.atenciones = []

    def agregar_atencion(self, fecha, especialidad, medico, diagnostico, tratamiento):
        atencion = {
            "fecha": fecha,
            "especialidad": especialidad,
            "medico": medico,
            "diagnostico": diagnostico,
            "tratamiento": tratamiento
        }
        self.atenciones.append(atencion)

    def total_atenciones(self):
        return len(self.atenciones)

    def ultimo_diagnostico(self):
        if len(self.atenciones) == 0:
            return "Sin atenciones registradas"
        ultima = self.atenciones[-1]
        return ultima["diagnostico"]

    def mostrar(self):
        print("HISTORIAL CLINICO DEL PACIENTE " + self.codigo_paciente)
        if len(self.atenciones) == 0:
            print("   (Sin atenciones registradas)")
        else:
            numero = 1
            for a in self.atenciones:
                print("   " + str(numero) + ". " + a["fecha"] + " | " + a["especialidad"] +
                      " | Dx: " + a["diagnostico"] + " | Tto: " + a["tratamiento"])
                numero = numero + 1
        print("   Total de atenciones: " + str(self.total_atenciones()))


print("Clase Historial creada.")

Clase Historial creada.


## 1.2 Clase base `Persona`

Reúne los cuatro datos que comparten pacientes y médicos. Se usan **getters** para leer y **setters** para modificar: el setter revisa el dato antes de guardarlo, por eso `set_telefono("123")` no modifica nada y avisa del error.

El método `mostrar_informacion()` se escribe aquí de forma general y las clases hijas lo van a **sobrescribir**.

In [ ]:
# ------------------------------------------------------------
#  Clase base Persona
#  Integrante: Anaya Yarleque, Nahomi Alessandra
# ------------------------------------------------------------
# Reune los datos que comparten pacientes y medicos. De ella van
# a HEREDAR las clases Paciente y Medico.


class Persona:
    def __init__(self, nombre, apellido, dni, telefono):
        self.nombre = nombre
        self.apellido = apellido
        self.dni = dni
        self.telefono = telefono

    # ---------- Getters: leer los datos ----------
    def get_nombre_completo(self):
        return self.nombre + " " + self.apellido

    def get_dni(self):
        return self.dni

    # ---------- Setter: modificar con validacion ----------
    def set_telefono(self, nuevo_telefono):
        """Cambia el telefono solo si tiene 9 digitos numericos."""
        if len(nuevo_telefono) == 9 and nuevo_telefono.isdigit():
            self.telefono = nuevo_telefono
            print("Telefono actualizado a " + nuevo_telefono)
        else:
            print("ERROR: el telefono debe tener 9 digitos numericos.")

    # ---------- Metodo que las clases hijas van a sobrescribir ----------
    def mostrar_informacion(self):
        return "[PERSONA]  " + self.get_nombre_completo() + " | DNI: " + self.dni

    def __str__(self):
        return self.get_nombre_completo()


print("Clase Persona creada.")

Clase Persona creada.


## 1.3 Clase `Paciente` (hereda de `Persona`)

Tres mecanismos trabajando en el mismo constructor:

1. **`super().__init__(...)`** llama al constructor del padre. Sin esa línea habría que repetir las cuatro asignaciones de los datos personales en cada clase hija.
2. **`contador`** es un atributo de clase que se incrementa cada vez que nace un paciente, y genera los códigos `P001`, `P002`, `P003`.
3. **`self.historial = Historial(...)`** es **composición**: un objeto que vive dentro de otro.

In [ ]:
# ------------------------------------------------------------
#  Clase Paciente (hereda de Persona)
#  Integrante: Anaya Yarleque, Nahomi Alessandra
# ------------------------------------------------------------


class Paciente(Persona):
    # Atributo de CLASE: genera los codigos P001, P002, P003...
    contador = 0

    def __init__(self, nombre, apellido, dni, telefono, edad, tiene_seguro=False):
        # super() reutiliza el constructor del padre
        super().__init__(nombre, apellido, dni, telefono)

        self.edad = edad
        self.tiene_seguro = tiene_seguro

        Paciente.contador = Paciente.contador + 1
        self.codigo = "P" + str(Paciente.contador).zfill(3)

        # COMPOSICION: cada paciente nace con su propio historial
        self.historial = Historial(self.codigo)

    def get_codigo(self):
        return self.codigo

    def set_edad(self, nueva_edad):
        if nueva_edad > 0 and nueva_edad < 120:
            self.edad = nueva_edad
        else:
            print("ERROR: la edad debe estar entre 1 y 119 anios.")

    def es_adulto_mayor(self):
        """Los pacientes de 60 anios a mas tienen atencion preferente."""
        return self.edad >= 60

    def a_diccionario(self):
        """Convierte el objeto en un diccionario, para usarlo con pandas."""
        return {
            "codigo": self.codigo,
            "nombre": self.get_nombre_completo(),
            "edad": self.edad,
            "tiene_seguro": self.tiene_seguro,
            "atenciones": self.historial.total_atenciones()
        }

    # ---------- Sobrescritura (override) ----------
    def mostrar_informacion(self):
        if self.tiene_seguro:
            seguro = "SI"
        else:
            seguro = "NO"

        texto = ("[PACIENTE] " + self.codigo + " | " + self.get_nombre_completo() +
                 " | Edad: " + str(self.edad) + " | Seguro: " + seguro)

        if self.es_adulto_mayor():
            texto = texto + " | Atencion preferente"
        return texto

    def __str__(self):
        return self.codigo + " - " + self.get_nombre_completo()


print("Clase Paciente creada.")

Clase Paciente creada.
